In [1]:
# Accuracy         →  % of predictions correct overall
#                     misleading with class imbalance

# Precision        →  of all times we predicted UP, how many were actually UP?
#                     "when we say up, are we right?"

# Recall           →  of all actual UP days, how many did we catch?
#                     "do we find most of the up days?"

# F1 Score         →  harmonic mean of precision and recall
#                     balances both concerns
#                     best single metric for imbalanced data

# Confusion Matrix →  shows exactly where the model is right and wrong
                    
#                     Predicted DOWN  Predicted UP
# Actual DOWN  [  True Negative  |  False Positive  ]
# Actual UP    [  False Negative |  True Positive   ]
import pandas as pd
import numpy as np
import psycopg2
#provides an API for logging and loading scikit-learn models
import mlflow
import mlflow.sklearn

#we will be using random forest for the ml traning 
#when we train multiple decision tree on random subsets of the data and look for major trends
#this ensure the model can predict without overfitting to the current data
from sklearn.ensemble import RandomForestClassifier
# IF a model trains on feburary and that data is tested on jan then that would be like cheating 
#because the model is alreadey know so to make sure that the model trains on older data first we user timeseriessplit
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nepse_price_direction")
load_dotenv()

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD")
)

df = pd.read_sql("SELECT * FROM gold_nepse_features", conn)
print(f"Loaded {len(df)} rows")

C:\Users\utsab\AppData\Local\Temp\ipykernel_26120\2507032838.py:51: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM gold_nepse_features", conn)


Loaded 242261 rows


In [2]:
# filter to 2025 onwards
df_model = df[pd.to_datetime(df['scraped_date']).dt.year >= 2025].copy()

# filter stocks with sufficient data
trading_days = df_model.groupby('symbol')['scraped_date'].nunique()
sufficient = trading_days[trading_days >= 100].index
df_model = df_model[df_model['symbol'].isin(sufficient)]

# create ratio features
df_model['price_vs_7day'] = df_model['ltp'] / df_model['avg_7_day']
df_model['trend_strength'] = df_model['avg_7_day'] / df_model['avg_30_day']
df_model['price_position'] = (
    (df_model['ltp'] - df_model['min_30_day']) /
    (df_model['max_30_day'] - df_model['min_30_day'] + 0.0001)
)

# create 5-day forward target
df_model = df_model.sort_values(['symbol', 'scraped_date'])
df_model['ltp_5day_forward'] = df_model.groupby('symbol')['ltp'].shift(-5)
df_model['target'] = (df_model['ltp_5day_forward'] > df_model['ltp']).astype(int)

# drop nulls
features = ['price_vs_7day', 'trend_strength', 'price_position']
df_model = df_model.dropna(subset=features + ['target'])

print(f"Shape: {df_model.shape}")
print(f"Class balance: {df_model['target'].mean():.2%} up")
print(df_model[features].describe())

Shape: (144004, 14)
Class balance: 43.78% up
       price_vs_7day  trend_strength  price_position
count  144004.000000   144004.000000   144004.000000
mean        1.000074        1.000566        0.445266
std         0.024697        0.044681        0.343239
min         0.584295        0.573089        0.000000
25%         0.989276        0.980669        0.119120
50%         0.998800        0.998214        0.413109
75%         1.008603        1.016532        0.742855
max         1.293443        1.828696        1.000000


In [3]:
# sort by date for time series split
df_model = df_model.sort_values('scraped_date')

X = df_model[features].values
y = df_model['target'].values

# time series cross validation
# creates 5 section of split using time 
tscv = TimeSeriesSplit(n_splits=5)

# mlflow experiment
mlflow.set_experiment("nepse_price_direction")

with mlflow.start_run(run_name="random_forest_baseline"):

    # log parameters
    params = {
        # build 100 trees
        "n_estimators": 100,
        # go to 10 level of depth
        "max_depth": 10,
        # make it balance since there are more down days than up days
        "class_weight": "balanced",
         #sets the random seed. Without this results change every run making it impossible to reproduce or compare. 42 is convention — any number works.
        "random_state": 42,
        "n_splits": 5
    }
    mlflow.log_params(params)

    model = RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        class_weight=params["class_weight"],
        random_state=params["random_state"],
        #use all core present
        n_jobs=-1
    )

    # cross validation scores
    fold_f1_scores = []
    fold_accuracies = []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        f1 = f1_score(y_test, y_pred, average='weighted')
        accuracy = (y_pred == y_test).mean()

        fold_f1_scores.append(f1)
        fold_accuracies.append(accuracy)

        print(f"Fold {fold+1}: accuracy={accuracy:.3f} f1={f1:.3f}")

    # average metrics across folds
    mean_f1 = np.mean(fold_f1_scores)
    mean_accuracy = np.mean(fold_accuracies)

    mlflow.log_metric("mean_f1", mean_f1)
    mlflow.log_metric("mean_accuracy", mean_accuracy)

    # train final model on all data
    model.fit(X, y)
    mlflow.sklearn.log_model(model, "random_forest_model")
    #Accuracy = (TP + TN) / Total
    #Precision = TP / (TP + FP) for up 
    #trendsRecall = TP / (TP + FN) how many time the model failed t predict up lost opportunites
    print(f"\nMean Accuracy: {mean_accuracy:.3f}")
    print(f"Mean F1 Score: {mean_f1:.3f}")

    # classification report on last fold
    print(f"\nClassification Report (last fold):")
    print(classification_report(y_test, y_pred, 
                                target_names=['Down', 'Up']))
#     If we want high precision (be sure before predicting UP):
# → Model only predicts UP when very confident
# → Misses many actual UP days (low recall)
# → But when it does predict UP, usually right

# If we want high recall (catch all UP days):
# → Model predicts UP frequently
# → Catches most actual UP days
# → But many false alarms (low precision, loses money on bad trades)

#important
# F1 = 2 × (Precision × Recall) / (Precision + Recall)
# Regular average of 0.9 precision and 0.1 recall:
# (0.9 + 0.1) / 2 = 0.5  ← looks okay

# Harmonic mean:
# 2 × (0.9 × 0.1) / (0.9 + 0.1) = 0.18  ← reveals the imbalance

# Harmonic mean punishes extreme imbalance between precision and recall
# A model that's great at one but terrible at the other gets a low F1

Fold 1: accuracy=0.527 f1=0.528
Fold 2: accuracy=0.498 f1=0.498
Fold 3: accuracy=0.458 f1=0.470
Fold 4: accuracy=0.548 f1=0.548
Fold 5: accuracy=0.522 f1=0.522


2026/04/06 14:09:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/06 14:09:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/06 14:09:37 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



Mean Accuracy: 0.510
Mean F1 Score: 0.513

Classification Report (last fold):
              precision    recall  f1-score   support

        Down       0.57      0.57      0.57     13275
          Up       0.46      0.46      0.46     10725

    accuracy                           0.52     24000
   macro avg       0.52      0.52      0.52     24000
weighted avg       0.52      0.52      0.52     24000



In [4]:
# Cell 4 — engineer additional features
# this data set has 6 features while the prev one had only 3 
df_model2 = df_model.copy()

# momentum features
df_model2 = df_model2.sort_values(['symbol', 'scraped_date'])

# 1. price rate of change over 5 days
# short term change
df_model2['roc_5'] = df_model2.groupby('symbol')['ltp'].transform(
    lambda x: x.pct_change(5)
)

# 2. price rate of change over 10 days  
df_model2['roc_10'] = df_model2.groupby('symbol')['ltp'].transform(
    lambda x: x.pct_change(10)
)

# 3. volatility — standard deviation of last 7 days returns
df_model2['returns'] = df_model2.groupby('symbol')['ltp'].transform(
    lambda x: x.pct_change(1)
)
df_model2['volatility_7'] = df_model2.groupby('symbol')['returns'].transform(
    lambda x: x.rolling(7).std()
)

# 4. volume trend — is volume increasing or decreasing?
# only available in 2025+ data
df_model2['avg_7day'] = df_model2['avg_7_day']

# drop nulls from new features
features_v2 = [
    'price_vs_7day',
    'trend_strength', 
    'price_position',
    'roc_5',
    'roc_10',
    'volatility_7'
]

df_model2 = df_model2.dropna(subset=features_v2 + ['target'])

print(f"Shape after adding features: {df_model2.shape}")
print(f"Class balance: {df_model2['target'].mean():.2%}")
print(df_model2[features_v2].describe())
# add this quick check
print(df_model2[features_v2].corr().round(2))

Shape after adding features: (140374, 19)
Class balance: 43.61%
       price_vs_7day  trend_strength  price_position          roc_5  \
count  140374.000000   140374.000000   140374.000000  140374.000000   
mean        0.999788        1.000785        0.445667       0.000307   
std         0.023989        0.044807        0.343330       0.039991   
min         0.584295        0.573089        0.000000      -0.460646   
25%         0.989219        0.980835        0.119175      -0.018299   
50%         0.998728        0.998342        0.413898      -0.001203   
75%         1.008377        1.016819        0.743703       0.015212   
max         1.293443        1.828696        1.000000       0.610389   

              roc_10   volatility_7  
count  140374.000000  140374.000000  
mean        0.001855       0.013965  
std         0.062592       0.010484  
min        -0.500585       0.000000  
25%        -0.025797       0.007274  
50%        -0.001942       0.011450  
75%         0.023256       0.0

In [5]:
# Cell 5 — refined feature set
# we drop roc_5 since it is somewhat redundant high correlation with price_vs7days and roc_10
features_v2_refined = [
    'price_vs_7day',    # short term position
    'trend_strength',   # momentum direction
    'price_position',   # 30-day range position
    'roc_10',           # medium term momentum
    'volatility_7'      # recent chaos/stability
]

print("Refined correlation matrix:")
print(df_model2[features_v2_refined].corr().round(2))

Refined correlation matrix:
                price_vs_7day  trend_strength  price_position  roc_10  \
price_vs_7day            1.00            0.29            0.54    0.71   
trend_strength           0.29            1.00            0.61    0.68   
price_position           0.54            0.61            1.00    0.58   
roc_10                   0.71            0.68            0.58    1.00   
volatility_7             0.07            0.25            0.18    0.21   

                volatility_7  
price_vs_7day           0.07  
trend_strength          0.25  
price_position          0.18  
roc_10                  0.21  
volatility_7            1.00  


In [6]:
# Cell 6 — second experiment with refined features
features_v2_refined = [
    'price_vs_7day',
    'trend_strength',
    'price_position',
    'roc_10',
    'volatility_7'
]

df_v2 = df_model2.dropna(subset=features_v2_refined + ['target'])
df_v2 = df_v2.sort_values('scraped_date')

X2 = df_v2[features_v2_refined].values
y2 = df_v2['target'].values

tscv = TimeSeriesSplit(n_splits=5)

with mlflow.start_run(run_name="random_forest_v2_more_features"):

    params = {
        "n_estimators": 100,
        "max_depth": 10,
        "class_weight": "balanced",
        "random_state": 42,
        "n_splits": 5,
        "features": str(features_v2_refined)
    }
    mlflow.log_params(params)

    model_v2 = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    fold_f1_scores = []
    fold_accuracies = []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X2)):
        X_train, X_test = X2[train_idx], X2[test_idx]
        y_train, y_test = y2[train_idx], y2[test_idx]

        model_v2.fit(X_train, y_train)
        y_pred = model_v2.predict(X_test)

        f1 = f1_score(y_test, y_pred, average='weighted')
        accuracy = (y_pred == y_test).mean()

        fold_f1_scores.append(f1)
        fold_accuracies.append(accuracy)

        print(f"Fold {fold+1}: accuracy={accuracy:.3f} f1={f1:.3f}")

    mean_f1 = np.mean(fold_f1_scores)
    mean_accuracy = np.mean(fold_accuracies)

    mlflow.log_metric("mean_f1", mean_f1)
    mlflow.log_metric("mean_accuracy", mean_accuracy)

    # train final model on all data
    model_v2.fit(X2, y2)
    mlflow.sklearn.log_model(model_v2, "random_forest_v2")

    print(f"\nMean Accuracy: {mean_accuracy:.3f}")
    print(f"Mean F1: {mean_f1:.3f}")

    # classification report
    print(f"\nClassification Report (last fold):")
    print(classification_report(y_test, y_pred,
                                target_names=['Down', 'Up']))

    # feature importance
    importances = model_v2.feature_importances_
    for feat, imp in zip(features_v2_refined, importances):
        print(f"  {feat}: {imp:.3f}")
    mlflow.log_dict(
        dict(zip(features_v2_refined, importances.tolist())),
        "feature_importance.json"
    )

Fold 1: accuracy=0.537 f1=0.539
Fold 2: accuracy=0.512 f1=0.512
Fold 3: accuracy=0.484 f1=0.499
Fold 4: accuracy=0.553 f1=0.552
Fold 5: accuracy=0.544 f1=0.545


2026/04/06 14:09:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/06 14:09:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/06 14:09:56 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



Mean Accuracy: 0.526
Mean F1: 0.529

Classification Report (last fold):
              precision    recall  f1-score   support

        Down       0.60      0.58      0.59     13134
          Up       0.48      0.50      0.49     10261

    accuracy                           0.54     23395
   macro avg       0.54      0.54      0.54     23395
weighted avg       0.55      0.54      0.54     23395

  price_vs_7day: 0.151
  trend_strength: 0.177
  price_position: 0.158
  roc_10: 0.214
  volatility_7: 0.300
